# 11. Structured Streaming: Real-Time Stream Processing

In modern data architectures (IoT sensors, financial fraud detection, live user activity), data arrives continuously in real-time. Spark's **Structured Streaming** treats streaming data as an **unbounded table**!

### Topics Covered:
1. **Batch vs. Stream Processing**
2. **The "Unbounded Table" Philosophy**
3. **The 3 Core Components: Source $\rightarrow$ Transformation $\rightarrow$ Sink**
4. **Creating a Streaming DataFrame using the `rate` source**
5. **Real-time Transformations & Aggregations**
6. **Output Modes: `append`, `complete`, `update`**
7. **Writing to an In-Memory Sink & Querying Live Data with SQL**
8. **Triggers, Watermarking & Checkpointing Concepts**
9. **Safely stopping the streaming query and SparkSession**


---
## 1. Initializing SparkSession


In [1]:


from pyspark.sql import SparkSession
from pyspark.sql.functions import col, current_timestamp, window, count, expr
import time

spark = SparkSession.builder \
    .appName("11_Structured_Streaming") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print("SparkSession initialized for Structured Streaming!")


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/11/11 18:58:04 WARN Utils: Your hostname, Tiger, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/11/11 18:58:04 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/11 18:58:05 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


SparkSession initialized for Structured Streaming!


---
## 2. The Unbounded Table Concept

In Structured Streaming, you write the exact same DataFrame queries as you would in batch mode.
* As new data arrives, Spark automatically appends it as new rows to an **Unbounded Table**.
* Spark generates an incremental execution plan that processes only newly arrived data.

```
Incoming Stream (Events over time) ──►  [ Unbounded Table ]  ──►  Query Engine  ──►  Output Sink
                                            ▲
                                        Appends new rows
```


---
## 3. Creating a Live Stream with the `rate` Source

Spark includes a built-in `rate` stream source that automatically generates sequential numbers with timestamps at a specified rate (rows per second). It is ideal for testing!


In [2]:
# Create a streaming DataFrame generating 2 rows per second
streaming_df = spark.readStream \
    .format("rate") \
    .option("rowsPerSecond", 2) \
    .load()

# Check if the DataFrame is streaming
print("Is DataFrame a real-time stream?", streaming_df.isStreaming)
streaming_df.printSchema()


Is DataFrame a real-time stream? True
root
 |-- timestamp: timestamp (nullable = true)
 |-- value: long (nullable = true)



---
## 4. Applying Streaming Transformations

Transformations on streaming DataFrames look just like standard DataFrame operations.


In [3]:
# Add custom columns to the stream
processed_stream = streaming_df.withColumn("square", col("value") * col("value")) \
                               .withColumn("is_even", col("value") % 2 == 0)

processed_stream.printSchema()


root
 |-- timestamp: timestamp (nullable = true)
 |-- value: long (nullable = true)
 |-- square: long (nullable = true)
 |-- is_even: boolean (nullable = true)



---
## 5. Output Modes Explained

When saving output from a streaming query, you must choose an **Output Mode**:

| Output Mode | Behavior | When to Use |
| :--- | :--- | :--- |
| **`append`** | Only new rows written since the last trigger are emitted. | Non-aggregated queries or watermarked streaming joins |
| **`complete`** | The entire updated result table is emitted every time. | Aggregation queries (e.g. running counts or sums) |
| **`update`** | Only rows that changed/updated since the last trigger are emitted. | Updating existing state without emitting unchanged rows |


---
## 6. Running a Live Stream to an In-Memory Sink

Let's start the stream, write it into an in-memory temporary table named `live_stream_table`, and query it using standard SQL!


In [4]:
# Start streaming query using memory sink
query = processed_stream.writeStream \
    .format("memory") \
    .queryName("live_stream_table") \
    .outputMode("append") \
    .start()

print(f"Streaming query started! Query ID: {query.id}")
print(f"Is query actively running? {query.isActive}")


26/11/11 18:58:10 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-cf61363f-38c5-4ad6-a9df-388579e1a7a5. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/11/11 18:58:10 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


Streaming query started! Query ID: 54e01a03-588e-42d0-ada1-2ab4e079ad5b
Is query actively running? True


In [5]:
# Let the stream run for 5 seconds to accumulate data
print("Waiting 5 seconds for streaming data to arrive...")
time.sleep(5)

# Query the in-memory streaming table using Spark SQL!
print("Live Stream Data Snapshot 1:")
spark.sql("SELECT * FROM live_stream_table ORDER BY timestamp DESC LIMIT 5").show()


Waiting 5 seconds for streaming data to arrive...
Live Stream Data Snapshot 1:
+--------------------+-----+------+-------+
|           timestamp|value|square|is_even|
+--------------------+-----+------+-------+
|2026-11-11 18:58:...|    9|    81|  false|
|2026-11-11 18:58:...|    8|    64|   true|
|2026-11-11 18:58:...|    7|    49|  false|
|2026-11-11 18:58:...|    6|    36|   true|
|2026-11-11 18:58:...|    5|    25|  false|
+--------------------+-----+------+-------+



In [6]:
# Wait another 4 seconds and query again to see new live rows!
print("Waiting another 4 seconds...")
time.sleep(4)

print("Live Stream Data Snapshot 2 (Notice new rows have arrived!):")
spark.sql("SELECT count(*) as total_events_received FROM live_stream_table").show()
spark.sql("SELECT * FROM live_stream_table ORDER BY timestamp DESC LIMIT 5").show()


Waiting another 4 seconds...
Live Stream Data Snapshot 2 (Notice new rows have arrived!):
+---------------------+
|total_events_received|
+---------------------+
|                   20|
+---------------------+

+--------------------+-----+------+-------+
|           timestamp|value|square|is_even|
+--------------------+-----+------+-------+
|2026-11-11 18:58:...|   19|   361|  false|
|2026-11-11 18:58:...|   18|   324|   true|
|2026-11-11 18:58:...|   17|   289|  false|
|2026-11-11 18:58:...|   16|   256|   true|
|2026-11-11 18:58:...|   15|   225|  false|
+--------------------+-----+------+-------+



---
## 7. Fault Tolerance: Checkpointing & Watermarking

* **Checkpointing (`checkpointLocation`)**:
  * In production, you specify a directory on cloud storage (S3/ADLS/GCS/HDFS) where Spark records its stream progress and offsets.
  * If a machine crashes or restarts, Spark reads the checkpoint and resumes exactly where it left off (Zero data loss, Exactly-Once semantics!).
* **Watermarking (`withWatermark("timestamp", "10 minutes")`)**:
  * In event-time streaming, events may arrive late due to network lag.
  * Watermarking tells Spark how late data can arrive before it is discarded: *"Wait up to 10 minutes for late events, then finalize the window."*


---
## 8. Safely Stopping the Streaming Query

Always stop active streaming queries before stopping the SparkSession!


In [7]:
# Stop the streaming query
query.stop()
print("Streaming query successfully stopped!")
print("Is query active now?", query.isActive)


Streaming query successfully stopped!
Is query active now? False


26/11/11 18:58:21 WARN DAGScheduler: Failed to cancel job group d5c7dba7-661f-4d0e-a8a1-db893047f381. Cannot find active jobs for it.
26/11/11 18:58:21 WARN DAGScheduler: Failed to cancel job group d5c7dba7-661f-4d0e-a8a1-db893047f381. Cannot find active jobs for it.


---
## 9. Summary of Notebook 11
* Structured Streaming treats incoming data as an unbounded table.
* The streaming pipeline consists of: **Source $\rightarrow$ Transformation $\rightarrow$ Sink**.
* Understand the 3 output modes: `append`, `complete`, and `update`.
* Checkpointing provides fault tolerance and exactly-once processing guarantees.
* Always stop the streaming query with `query.stop()` before stopping Spark.

---
## ⚠️ CRITICAL STEP: Stopping the SparkSession
Always release RAM and JVM background processes before moving to the next notebook.


In [8]:
# Stop SparkSession to free memory
spark.stop()
print("SparkSession stopped successfully! RAM released.")


SparkSession stopped successfully! RAM released.
